In [1]:
import pandas as pd
import os
import requests
import zipfile
import io
import matplotlib.pyplot as plt
import seaborn as sns
import platform

In [4]:
def get_qualtrics_survey(dir_save_survey, survey_id):
    """ automatically query the qualtrics survey data
    guide https://community.alteryx.com/t5/Alteryx-Designer-Discussions/Python-Tool-Downloading-Qualtrics-Survey-Data-using-Python-API/td-p/304898 """

    # Setting user Parameters
    api_token = "QUahqaqAx3GWF0pswKjbYYR37cil6xd0vN2zpB85" # your API ID
    file_format = "csv"
    data_center = 'uofsc.sjc1' # "<Organization ID>.<Datacenter ID>"

    # Setting static parameters
    request_check_progress = 0
    progress_status = "in progress"
    base_url = "https://{0}.qualtrics.com/API/v3/responseexports/".format(data_center)
    headers = {
        "content-type": "application/json",
        "x-api-token": api_token,
    }

    # Step 1: Creating Data Export
    download_request_url = base_url
    download_request_payload = '{"format":"' + file_format + '","surveyId":"' + survey_id + '"}' # you can set useLabels:True to get responses in text format
    download_request_response = requests.request("POST", download_request_url, data=download_request_payload, headers=headers)
    progress_id = download_request_response.json()["result"]["id"]
    # print(download_request_response.text)

    # Step 2: Checking on Data Export Progress and waiting until export is ready
    while request_check_progress < 100 and progress_status != "complete":
        request_check_url = base_url + progress_id
        request_check_response = requests.request("GET", request_check_url, headers=headers)
        request_check_progress = request_check_response.json()["result"]["percentComplete"]

    # Step 3: Downloading file
    request_download_url = base_url + progress_id + '/file'
    request_download = requests.request("GET", request_download_url, headers=headers, stream=True)

    # Step 4: Unzipping the file
    zipfile.ZipFile(io.BytesIO(request_download.content)).extractall(dir_save_survey)
    print('Downloaded qualtrics survey',survey_id)
def preprocess_segments(story,shuffle):
    # Dir_segments = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github\data\segmentation'
    # file = f"segment_{story}_formatted.csv"
    # filepath = os.path.join(Dir_segments,file)
    if platform.system() == 'Linux':
        Dir_timestamps = r'/work/xy6/Project/ABC/github/derivatives/behavioral/timestamps/timestamps_segment/' # HPC
    else:
        Dir_timestamps = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github\derivatives\behavioral\timestamps\timestamps_segment'
    
    file = f"timestamps_{story}.csv"
    filepath = os.path.join(Dir_timestamps,file)
    df = pd.read_csv(filepath)
    df['segment'] = df['segment'].str.strip().replace('\n','')
    df['idx_raw'] = df['idx']
    df['segment_raw'] = df['segment']
    # remove special symbols
    for i,row in df.iterrows():
        line = row['segment']
        while (line[-1] in '",.…!?”“') | (line.endswith(' ')) :
           line = line[0:-1]
        df.loc[i,'segment'] = line

    # shuffle if needed
    if shuffle == 1:
        df.sample(frac=1).reset_index(drop=True)
        df = df.sample(frac=1).reset_index(drop=True)
        df['idx'] = df.index

    # add index
    df['text'] = df['idx']+1
    df['text'] = df['text'].astype(str)
    df['text'] = '#####'+df['text'] + ' ' + df['segment']+'#####\n'

    N_segments = df.shape[0]
    lines_story = ' '.join(df['text'].values)
    return(df,lines_story,N_segments)
    
# def format_ratings_human(data):
#     df_ratings_raw = []
#     for modality in ['A','V']:
#         # ImagineTime
#         df = pd.DataFrame(data[[x for x in data.columns if x.startswith(f'Imagine_timing_{modality}_3')]].values)
#         df.columns = [f"x{i+1}" for i in range(df.shape[1])]
#         df['subid'] = data['ResponseID']
#         df['subid_idx'] = data.index
#         df['modality'] = modality
#         df['rating'] = 'ImagineTime'
#         df_ratings_raw.append(df)

#         # valence-arousal grid
#         for var, label in zip(['V','A'],['_x','_y']):
#             df = pd.DataFrame(data[[x for x in data.columns if x.startswith(f'Grid_{modality}') and label in x]].values)
#             df.columns = [f"x{i+1}" for i in range(df.shape[1])]
#             df['subid'] = data['ResponseID']
#             df['subid_idx'] = data.index
#             df['modality'] = modality
#             df['rating'] = var
#             df_ratings_raw.append(df)

#         # grid RT
#         df = pd.DataFrame(data[[x for x in data.columns if x.startswith(f'Grid_timing_{modality}_3')]].values)
#         df.columns = [f"x{i+1}" for i in range(df.shape[1])]
#         df['subid'] = data['ResponseID']
#         df['subid_idx'] = data.index
#         df['modality'] = modality
#         df['rating'] = 'RT_grid'
#         df_ratings_raw.append(df)

#         # behavioral ratings
#         for var in ['Vividness','Difficulty','Familiarity']:
#             df = pd.DataFrame(data[data[[x for x in data.columns if x.startswith(f'{var}_{modality}')]].columns[3:]].values)
#             df.columns = [f"x{i+1}" for i in range(df.shape[1])]
#             df['subid'] = data['ResponseID']
#             df['subid_idx'] = data.index
#             df['modality'] = modality
#             df['rating'] = var
#             df_ratings_raw.append(df)

#     df_ratings_raw = pd.concat(df_ratings_raw,ignore_index=True)
#     return(df_ratings_raw)

In [13]:
if platform.system() == 'Linux':
    Dir_github = r'/work/xy6/Project/ABC/github/' # HPC
else:
    Dir_github = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github' # Local computer
repo = 'ABC_valence'
pipeline = 'getRatings_Human'

# Dir_working = os.path.join(Dir_github,repo,pipeline)
Dir_segments = os.path.join(Dir_github,'derivatives','behavioral','segmentation')
Dir_output = os.path.join(Dir_github,'derivatives','behavioral','ratings','human','raw')
if not os.path.exists(Dir_output):
    os.makedirs(Dir_output)
surveyID = 'SV_3rC6kGBNBknyZg2'
get_qualtrics_survey(Dir_output, surveyID)

Downloaded qualtrics survey SV_3rC6kGBNBknyZg2


In [11]:
def get_rating_human(Dir_github,project_name,List_story):
    dict_vars = {'age':'Q1','gender':'Q3',
             'S1_vivid':'Q22_4','S2_vivid':'Q105_4','Q3_vivid':'Q108_4',
             'S1_Q1':'Q55','S1_Q2':'Q65','S1_Q3':'Q66','S1_Q4':'Q67',
             'S2_Q1':'Q61','S2_Q2':'Q62','S2_Q3':'Q63','S2_Q4':'Q64',
             'S3_Q1':'Q56','S3_Q2':'Q58','S3_Q3':'Q59','S3_Q4':'Q60',
             }
    
    Dir_output = os.path.join(Dir_github,'derivatives','behavioral','ratings','human','raw')
    if not os.path.exists(Dir_output):
        os.makedirs(Dir_output)
    filepath = os.path.join(Dir_output,f'{project_name}.csv')
    data_raw = pd.read_csv(filepath)
    data = data_raw.loc[data_raw['Finished']=='1'].reset_index(drop=True) # only get the participants who have finished the entire experiment
    # data['duration'] = data['EndDate'].astype('datetime64[m]')-data['StartDate'].astype('datetime64[m]')
    data['duration'] = pd.to_datetime(data['EndDate'])-pd.to_datetime(data['StartDate']) 

    # other ratings
    data.rename({v: k for k, v in dict_vars.items()},inplace=True,axis=1)
    data[dict_vars.keys()]

    # get only VA
    df_ratings_raw = []
    for story in List_story:
        # df_tmp,lines_story,N_segments = preprocess_segments(story,shuffle=0)
        for var, label in zip(['V','A'],['_x','_y']):
            df = pd.DataFrame(data[[x for x in data.columns if x.startswith(f'rating_{story}') and label in x]].values)
            List_vars = [i for i in range(df.shape[1])]
            df.columns = List_vars
            df['subid'] = data['ResponseID']
            df['iteration'] = data.index
            df['story'] = story
            df['rating'] = var
            df['subid_idx'] = data['id']
            df['rater'] = 'human'
            df = pd.melt(df, id_vars=['story','rating','subid','subid_idx','iteration','rater'], value_vars=List_vars,var_name='segment').sort_values(['subid_idx','story','rating','segment','iteration','rater'])
            df_ratings_raw.append(df)
    df_ratings_raw = pd.concat(df_ratings_raw,ignore_index=True).dropna()
    
    df_ratings_raw['value'] = df_ratings_raw['value'].astype(float)
    df_ratings_raw.loc[df_ratings_raw['rating']=='V','value'] = (df_ratings_raw.loc[df_ratings_raw['rating']=='V','value']-240)/480*9
    df_ratings_raw.loc[df_ratings_raw['rating']=='A','value'] = (480-df_ratings_raw.loc[df_ratings_raw['rating']=='A','value'])/480*9
    df_ratings = df_ratings_raw.pivot(index=['story','rater','iteration','subid','subid_idx','segment'],columns = 'rating',values='value').reset_index()
    df_ratings.rename({'segment':'idx_raw'},axis=1,inplace=True)
    
    dfs_timestamps = []
    for story in List_story:
        df,lines_story,N_segments = preprocess_segments(story,shuffle=0)
        dfs_timestamps.append(df)
    dfs_timestamps = pd.concat(dfs_timestamps,ignore_index=True)
    
    
    df_ratings = pd.merge(left = df_ratings,right = dfs_timestamps, left_on=['story','idx_raw'],right_on=['story','idx_raw'],how='left')
    df_ratings.to_csv(os.path.join(Dir_output,f"rating_human_{project_name}.csv"),index=False)
    return(df_ratings)

In [14]:
project_name = 'ABCNaturalisticNarrativeAudio_sona_A'
List_story = ['FireSelected','Koko','Aqua']
df_ratings = get_rating_human(Dir_github,project_name,List_story)